<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 2:  Tareas Clásicas de NLP</h1>
    <h2>Traducción Automática (Embeddings)</h2>
</div>


Ahora implementaremos un sistema de traducción automática. Empecemos importando
las funciones requeridas!


```
nltk.download('stopwords')
nltk.download('twitter_samples')
```

In [1]:
# importar las librerías
import pdb
import pickle
import string
import time
import pandas as pd
import gensim
import matplotlib.pyplot as plt
import nltk
import numpy as np
import scipy
import sklearn
from gensim.models import KeyedVectors
from nltk.corpus import stopwords, twitter_samples
from nltk.tokenize import TweetTokenizer
from os import getcwd


# Embeddings para palabras en Inglés y Frances

Escribiremos un programa que traduzca del inglés al francés.

## Los datos

El conjunto de datos completo para las wordembeddings en inglés es de aproximadamente 3,64 gigabytes, y el francés
son de aproximadamente 629 megabytes. Trabajaremos con un conjunto más pequeño de datos, una muestra significativa.


* English embeddings from Google code archive word2vec
[look for GoogleNews-vectors-negative300.bin.gz](https://code.google.com/archive/p/word2vec/)
    * You'll need to unzip the file first.
* and the French embeddings from
[cross_lingual_text_classification](https://github.com/vjstark/crosslingual_text_classification).
    * in the terminal, type (in one line)
    `curl -o ./wiki.multi.fr.vec https://dl.fbaipublicfiles.com/arrival/vectors/wiki.multi.fr.vec`


#### Subconjunto de los datos




In [2]:
en_embeddings_subset = pickle.load(open("Data/en_embeddings.p", "rb"))
fr_embeddings_subset = pickle.load(open("Data/fr_embeddings.p", "rb"))

In [5]:
len(en_embeddings_subset['the'])

300

In [7]:
len(fr_embeddings_subset['était'])

300

#### Cargue dos diccionarios que mapean las palabras del inglés al francés
* Un diccionario de entrenamiento
* y un diccionario de prueba.

In [9]:
file = pd.read_csv('Data/en-fr.txt', delimiter = ' ')

In [10]:
file

,the,le
0,Ashgabat,ashgabat
1,the,les
2,the,la
3,and,et
4,was,fut
...,...,...
113281,taroudant,taroudant
113282,torra,torra
113283,nayer,nayer
113284,castagnola,castagnola


In [12]:
def get_dict(file_name):
    my_file = pd.read_csv(file_name, delimiter = ' ')
    etof = {}
    for i in range(len(my_file)):
        en = my_file.iloc[i,0]
        fr = my_file.iloc[i,1]
        etof[en] = fr

    return etof

In [13]:
#Diccionario de Ingles
en_fr_train = get_dict('Data/en-fr.train.txt')
en_fr_train

{'the': 'la',
 'and': 'et',
 'was': 'était',
 'for': 'pour',
 'that': 'cela',
 'with': 'avec',
 'from': 'depuis',
 'this': 'ce',
 'utc': 'tuc',
 'his': 'son',
 'not': 'pas',
 'are': 'sont',
 'talk': 'parlez',
 'which': 'lequel',
 'also': 'egalement',
 'were': 'étaient',
 'but': 'mais',
 'have': 'ont',
 'one': 'one',
 'new': 'nouveautés',
 'first': 'premiers',
 'page': 'page',
 'you': 'you',
 'they': 'eux',
 'had': 'avais',
 'article': 'article',
 'who': 'who',
 'all': 'all',
 'their': 'leurs',
 'there': 'là',
 'made': 'fabriqué',
 'its': 'son',
 'people': 'personnes',
 'may': 'peut',
 'after': 'aprés',
 'other': 'autres',
 'should': 'devrais',
 'two': 'deux',
 'score': 'partition',
 'her': 'her',
 'can': 'peut',
 'would': 'ferait',
 'more': 'plus',
 'she': 'elle',
 'when': 'quand',
 'time': 'heure',
 'team': 'equipe',
 'american': 'américains',
 'such': 'telles',
 'discussion': 'débat',
 'links': 'liens',
 'only': 'seule',
 'some': 'quelques',
 'see': 'vois',
 'united': 'unies',
 'year

In [17]:
#Diccionario de Frances
en_fr_test = get_dict('Data/en-fr.test.txt')
en_fr_test

{'torpedo': 'torpilles',
 'giovanni': 'giovanni',
 'chat': 'chat',
 'catholics': 'catholiques',
 'herald': 'herald',
 'chuck': 'chuck',
 'pit': 'fosse',
 'supplied': 'fournie',
 'optional': 'facultatives',
 'garrison': 'garnison',
 'sprint': 'sprint',
 'exile': 'exilés',
 'surprised': 'étonnée',
 'achievements': 'réalisations',
 'biblical': 'bibliques',
 'rebels': 'rebelles',
 'denis': 'denis',
 'geographical': 'géographique',
 'sit': 'sit',
 'alpine': 'alpine',
 'bills': 'factures',
 'glacier': 'glaciers',
 'binding': 'reliure',
 'indicating': 'indiquant',
 'estonia': 'estonie',
 'eating': 'manger',
 'saving': 'économiser',
 'chi': 'chi',
 'developer': 'développeurs',
 'indie': 'indie',
 'difficulties': 'difficultés',
 'doctrine': 'doctrine',
 'worn': 'porté',
 'fork': 'fourches',
 'simpson': 'simpson',
 'maintaining': 'maintenir',
 'theological': 'théologique',
 'upcoming': 'prochaines',
 'temporarily': 'momentanément',
 'hotels': 'hôtels',
 'edmonton': 'edmonton',
 'developments': '

#### Viendo los diccionarios de Inglés y Francés

* `en_fr_train` es un diccionario donde el key es la palabra en inglés y el valor es la palabra en francés.
```
{'the': 'la',
 'and': 'et',
 'was': 'était',
 'for': 'pour',
```

* `en_fr_test` es similar que `en_fr_train`, pero este es un conjunto de prueba. 

## Generando matrices de embeddings y transformación

####  Traducción de diccionario de inglés a francés mediante embeddings

Implementaremos la función `get_matrices`, que toma los datos cargados y retorna las matrices `X` y `Y`.

Entrada:
- `en_fr` : diccionario del Inglés a Francés
- `en_embeddings` : embeddings de palabras en inglés
- `fr_embeddings` : embeddings de palabras en Francés

Retorna:
- matrices `X` y `Y`, donde cada renglón en X es la palabra embebida para una plabra en inglés, y lo mismo con Y es la palabra embebida en francés.

<div style="width:image width px; font-size:100%; text-align:center;">
<img src='Figures/X_to_Y.jpg' alt="alternate text" width="width" height="height" style="width:800px;height:200px;" /> Figure 2 </div>

Utilice el diccionario `en_fr` para asegurarse de que la i-ésima fila de la matriz` X`
corresponde a la i-ésima fila de la matriz "Y".


In [23]:
def get_matrices(en_fr, french_vecs, english_vecs):

    X_l = list()
    Y_l = list()

    en_set = english_vecs.keys()
    fr_set = french_vecs.keys()

    french_words = set(en_fr.values())

    for en_word, fr_word in en_fr.items():

        if fr_word in fr_set and en_word in en_set:
            en_vec = english_vecs[en_word]
            fr_vec = french_vecs[fr_word]

            X_l.append(en_vec)
            Y_l.append(fr_vec)

    X = np.vstack(X_l)
    Y = np.vstack(Y_l)

    return X,Y    
        
    

Ahora usaremos la función `get_matrices ()` para obtener los conjuntos `X_train` e` Y_train`
de los embeddings de palabras en inglés y francés.


In [24]:
X_train, Y_train = get_matrices(en_fr_train, fr_embeddings_subset, en_embeddings_subset)

In [25]:
X_train

array([[ 0.08007812,  0.10498047,  0.04980469, ...,  0.00366211,
         0.04760742, -0.06884766],
       [ 0.02600098, -0.00189209,  0.18554688, ..., -0.12158203,
         0.22167969, -0.02197266],
       [-0.01177979, -0.04736328,  0.04467773, ...,  0.07128906,
        -0.03491211,  0.02416992],
       ...,
       [-0.17089844,  0.17871094, -0.06494141, ..., -0.10644531,
        -0.31640625, -0.09326172],
       [-0.21875   ,  0.09179688,  0.03637695, ..., -0.015625  ,
        -0.27148438,  0.14941406],
       [-0.00418091,  0.0703125 , -0.04516602, ..., -0.16015625,
         0.09326172, -0.15039062]], shape=(4932, 300), dtype=float32)

In [26]:
len(Y_train)

4932

In [27]:
Y_train

array([[-0.0061825 , -0.00094387, -0.00882648, ...,  0.111644  ,
        -0.0503964 , -0.0603421 ],
       [-0.0341354 ,  0.042414  , -0.0656882 , ..., -0.0539992 ,
         0.0371097 , -0.0433599 ],
       [ 0.0426481 ,  0.0395683 , -0.00825683, ...,  0.0295259 ,
         0.0713421 ,  0.0626402 ],
       ...,
       [ 0.0903279 , -0.108363  , -0.00956318, ..., -0.0337517 ,
        -0.00909727, -0.0503935 ],
       [-0.0753425 ,  0.0567269 ,  0.0230996 , ...,  0.0844913 ,
         0.0782853 , -0.0151161 ],
       [-0.0350154 , -0.0336099 , -0.0251155 , ...,  0.116152  ,
         0.110519  ,  0.0197631 ]], shape=(4932, 300), dtype=float32)


# Traductores

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Figures/e_to_f.jpg' alt="alternate text" width="width" height="height" style="width:700px;height:200px;" />  </div>

Necesitamos un programa que traduzca palabras del inglés al francés utilizando word embeddings y modelos de espacio vectorial.


##  Traducción como transformación lineal de embeddings
Dados los diccionarios de embeddings de palabras en inglés y francés, crearemos una matriz de transformación `R`
* Dada una embedding en inglés, $ \mathbf {e} $, podemos multiplicar $ \mathbf {eR} $ para obtener una nueva palabra (nuevo embedding) $ \mathbf {f} $.

    
* Después, Calcularemos los vecinos más cercanos a `f` en los embeddings francesas y recomendar la palabra que es más similar a los embeddings de palabras transformadas.

### Describir la traducción como el problema de minimización

Necesitamos encontrar una matriz `R` que minimice la siguiente ecuación.

$$\arg \min _{\mathbf{R}}\| \mathbf{X R} - \mathbf{Y}\|_{F}\tag{1} $$

### Norma de Frobenius

La norma de Frobenius de una matriz $ A $ (asumiendo que es de dimensión $ m, n $) se define como la raíz cuadrada de la suma de los cuadrados absolutos de sus elementos:

$$\|\mathbf{A}\|_{F} \equiv \sqrt{\sum_{i=1}^{m} \sum_{j=1}^{n}\left|a_{i j}\right|^{2}}\tag{2}$$

### Función de perdida

En las aplicaciones del mundo real, la pérdida de la norma Frobenius:

$$\| \mathbf{XR} - \mathbf{Y}\|_{F}$$

a menudo se reemplaza por su valor al cuadrado dividido por $ m $:

$$ \frac{1}{m} \|  \mathbf{X R} - \mathbf{Y} \|_{F}^{2}$$

donde $ m $ es el número de ejemplos (filas en $ \mathbf {X} $).

* Se encuentra la misma R cuando se usa esta función de pérdida en comparación con la norma de Frobenius original.
* La razón para tomar el cuadrado es que es más fácil calcular el gradiente del Frobenius al cuadrado.
* La razón para dividir entre $ m $ es que estamos más interesados en la pérdida promedio por inserción que en la pérdida de todo el conjunto de entrenamiento.
     * La pérdida de todo el conjunto de entrenamiento aumenta con más palabras (ejemplos de entrenamiento),
     por lo que tomar el promedio nos ayuda a rastrear la pérdida promedio independientemente del tamaño del conjunto de entrenamiento.



### Implementación del mecanismo de traducción

#### Calculando el loss
* La función de pérdida será la norma de Frobenoius al cuadrado de la diferencia entre
matriz y su aproximación, dividida por el número de ejemplos de entrenamiento $ m $.
* Su fórmula es:
$$ L(X, Y, R)=\frac{1}{m}\sum_{i=1}^{m} \sum_{j=1}^{n}\left( a_{i j} \right)^{2}$$

donde $a_{i j}$ es el valo de la $i$-ésimo renglón y $j$-ésima columna de la matriz $\mathbf{XR}-\mathbf{Y}$.

* Calcular la aproximación de `Y` mediante la matriz multiplicando` X` y `R`
* Calcular la diferencia `XR - Y`
* Calcular la norma de Frobenius al cuadrado de la diferencia y divídala por $ m $.

In [28]:
def compute_loss(X,Y,R):
    m = X.shape[0]

    diff = np.dot(X,R) - Y
    sum_diff_sq = np.sum(diff)

    loss = sum_diff_sq/m
    
    return loss
    


### Calculando el gradiente de la función loss con respecto a la matriz de transforación R

* Calcular el gradiente de la pérdida con respecto a la matriz de transformación "R".
* El gradiente nos da la dirección en la que debemos disminuir `R`
para minimizar la pérdida.
* $ m $ es el número de ejemplos de entrenamiento (número de filas en $ X $).
* La fórmula para el gradiente de la función de pérdida $ 𝐿 (𝑋, 𝑌, 𝑅) $ es:

$$\frac{d}{dR}𝐿(𝑋,𝑌,𝑅)=\frac{d}{dR}\Big(\frac{1}{m}\| X R -Y\|_{F}^{2}\Big) = \frac{2}{m}X^{T} (X R - Y)$$



In [36]:
def compute_gradient(X,Y,R):
    m = X.shape[0]

    gradient = (2/m)*(np.dot(X.T, np.dot(X,R)-Y))

    return gradient


### Encontrar la R óptima con el algoritmo de descenso de gradiente

#### Gradient descent


Pseudocódigo:
1. Calcular el gradiente $g$ del loss con respecto a la matriz $R$.
2. Update $R$ con la formula:
$$R_{\text{new}}= R_{\text{old}}-\alpha g$$

Donde $\alpha$ es el learning rate, que es un escalar.

#### Learning rate

* La tasa de aprendizaje o "tamaño de paso" $ \ alpha $ es un coeficiente que decide cuánto queremos cambiar $ R $ en cada paso.
* Si cambiamos $ R $ demasiado, podríamos saltarnos el óptimo dando un paso demasiado grande.
* Si solo hacemos pequeños cambios en $ R $, necesitaremos muchos pasos para alcanzar el óptimo.
* La tasa de aprendizaje $ \ alpha $ se usa para controlar esos cambios.
* Los valores de $ \ alpha $ se eligen dependiendo del problema, y usaremos `learning_rate` $ = 0.0003 $ como valor predeterminado para nuestro algoritmo.

In [37]:
def align_embeddings(X,Y, train_steps = 100, learning_rate = 0.0003):
    np.random.seed(129)

    R = np.random.rand(X.shape[1], X.shape[1])

    for i in range(train_steps):
        loss = compute_loss(X,Y,R)
        print(f' loss: {loss} en la iteración {i}')

        gradient = compute_gradient(X,Y,R)

        R =  R - learning_rate*gradient
    return R
    

In [39]:
np.random.seed(129)
m=10
n=5
X = np.random.rand(m,n)
Y = np.random.rand(m,n)*.1
#Probar la función
R = align_embeddings(X,Y, train_steps = 500, learning_rate = 0.0003)

 loss: 3.968070726141545 en la iteración 0
 loss: 3.9659212651935682 en la iteración 1
 loss: 3.9637729726883153 en la iteración 2
 loss: 3.9616258479903608 en la iteración 3
 loss: 3.9594798904646225 en la iteración 4
 loss: 3.957335099476368 en la iteración 5
 loss: 3.955191474391204 en la iteración 6
 loss: 3.9530490145750896 en la iteración 7
 loss: 3.9509077193943214 en la iteración 8
 loss: 3.948767588215547 en la iteración 9
 loss: 3.9466286204057517 en la iteración 10
 loss: 3.9444908153322706 en la iteración 11
 loss: 3.942354172362782 en la iteración 12
 loss: 3.940218690865307 en la iteración 13
 loss: 3.93808437020821 en la iteración 14
 loss: 3.935951209760197 en la iteración 15
 loss: 3.9338192088903234 en la iteración 16
 loss: 3.9316883669679825 en la iteración 17
 loss: 3.929558683362912 en la iteración 18
 loss: 3.927430157445194 en la iteración 19
 loss: 3.925302788585248 en la iteración 20
 loss: 3.923176576153844 en la iteración 21
 loss: 3.921051519522087 en la it

## Calcular la matriz de transformación R


In [40]:
R = align_embeddings(X_train,Y_train, train_steps = 500, learning_rate = 0.9)

 loss: -188.07721064858046 en la iteración 0
 loss: 18.618000852933555 en la iteración 1
 loss: -2.299496137461377 en la iteración 2
 loss: 3.5759996725358305 en la iteración 3
 loss: 3.161496053947133 en la iteración 4
 loss: 3.1135971477935938 en la iteración 5
 loss: 2.864464070649309 en la iteración 6
 loss: 2.625884518434399 en la iteración 7
 loss: 2.399575947307154 en la iteración 8
 loss: 2.193403709809634 en la iteración 9
 loss: 2.0068137260636347 en la iteración 10
 loss: 1.8383957933232713 en la iteración 11
 loss: 1.6863337193176695 en la iteración 12
 loss: 1.5488450013722312 en la iteración 13
 loss: 1.4242813659138378 en la iteración 14
 loss: 1.3111652363538888 en la iteración 15
 loss: 1.2081920844955139 en la iteración 16
 loss: 1.114219956679241 en la iteración 17
 loss: 1.0282536445742498 en la iteración 18
 loss: 0.9494273772088231 en la iteración 19
 loss: 0.8769879357308621 en la iteración 20
 loss: 0.8102791170252498 en la iteración 21
 loss: 0.7487279247584366

loss at iteration 377 is: 0.5686167334411015
loss at iteration 378 is: 0.5685466826988004
loss at iteration 379 is: 0.5684779546491274
loss at iteration 380 is: 0.5684105226586321
loss at iteration 381 is: 0.5683443606688969
loss at iteration 382 is: 0.5682794431832174
loss at iteration 383 is: 0.5682157452536102
loss at iteration 384 is: 0.5681532424681442
loss at iteration 385 is: 0.56809191093858
loss at iteration 386 is: 0.5680317272883223
loss at iteration 387 is: 0.5679726686406622
loss at iteration 388 is: 0.5679147126073114
loss at iteration 389 is: 0.567857837277216
loss at iteration 390 is: 0.5678020212056437
loss at iteration 391 is: 0.5677472434035448
loss at iteration 392 is: 0.5676934833271623
loss at iteration 393 is: 0.5676407208679026
loss at iteration 394 is: 0.5675889363424506
loss at iteration 395 is: 0.5675381104831243
loss at iteration 396 is: 0.5674882244284666
loss at iteration 397 is: 0.5674392597140602
loss at iteration 398 is: 0.5673911982635673
loss at itera

In [44]:
R.shape

(300, 300)

In [ ]:
# VF = e_iR

In [45]:
Yhat = np.dot(X_train,R)

In [46]:
Yhat

array([[-0.02340129, -0.0032963 , -0.02684166, ...,  0.02744616,
         0.01715794, -0.00366401],
       [-0.01362437, -0.00129046, -0.05520523, ...,  0.01677211,
         0.0459886 ,  0.00717925],
       [-0.01245856,  0.0137361 , -0.01382703, ...,  0.02461993,
         0.05008422, -0.01362901],
       ...,
       [ 0.02177541, -0.05538198, -0.03846587, ..., -0.01560807,
         0.03598485, -0.02774162],
       [-0.01879526, -0.00433944,  0.03008367, ...,  0.06957914,
        -0.00162228, -0.03386664],
       [-0.0270601 , -0.01752797, -0.02028583, ...,  0.04419445,
         0.05666284,  0.01831575]], shape=(4932, 300))


## Probando el traductor

### Algoritmo k-Nearest neighbors 

[k-Nearest neighbors algorithm](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) 
* k-NN es un método que toma un vector como entrada y encuentra los otros vectores en el conjunto de datos que están más cerca de él.
* La 'k' es el número de "vecinos más cercanos" a encontrar (por ejemplo, k = 2 encuentra los dos vecinos más cercanos).

### Buscando el embedding de la traducción 
Dado que estamos aproximando la función de traducción de los embeddings de inglés a francés mediante una matriz de transformación lineal $ \mathbf {R} $, la mayoría de las veces no obtendremos lel embedding de una palabra francesa cuando transformamos los embeddings $ \mathbf { e} $ de alguna palabra en inglés en particular en el espacio de embeddings francés.
* ¡Aquí es donde $ k $ -NN se vuelve realmente útil! Al usar $ 1 $ -NN con $ \mathbf {eR} $ como entrada, podemos buscar un embedding $ \mathbf {f} $ (como una fila) en la matriz $ \mathbf {Y} $ que es la más cercana a el vector transformado $ \mathbf {eR} $

### Similaridad por  Coseno
Similitud de coseno entre los vectores $ u $ y $ v $ calculada como el coseno del ángulo entre ellos.
La formula es
$$\cos(u,v)=\frac{u\cdot v}{\left\|u\right\|\left\|v\right\|}$$

* $\cos(u,v)$ = $1$ cuando $u$ y $v$ se encuentran en la misma línea y tienen la misma dirección.
* $\cos(u,v)$ es $-1$ Cuando ellas tienen direcciones exactamente opuestas.
* $\cos(u,v)$ es $0$ cuando los vectores son ortogonales (perpendiculares) entre sí.

#### Nota: La distancia y la similitud son cosas bastante opuestas.
* Podemos obtener la métrica de la distancia a partir de la similitud del coseno, pero la similitud del coseno no se puede usar directamente como la métrica de la distancia.
* Cuando la similitud del coseno aumenta (hacia $ 1 $), la "distancia" entre los dos vectores disminuye (hacia $ 0 $).
* Podemos definir la distancia del coseno entre $ u $ y $ v $ como
$$ d_{\text {cos}} (u, v) = 1- \cos(u, v) $$

In [48]:
def cosine_similarity(A,B):
    dot = np.dot(A, B)
    norma = np.linalg.norm(A)
    normb = np.linalg.norm(B)
    cos = dot / (norma * normb)
    return cos

Completando la función `nearest_neighbor()`:



In [51]:
def nearest_neighbor(v, candidates, k=1):

    similarity_l = []

    for row in candidates:
        cos_similarity = cosine_similarity(v, row)
        similarity_l.append(cos_similarity)

    sorted_ids = np.argsort(similarity_l)
    k_idx = sorted_ids[-k:]
    
    return k_idx, similarity_l
        


In [53]:
#Probando l función
v= np.array([1,0,1])
candidates=np.array([[1,0,5],
                     [-2,5,3],
                     [2,0,1],
                     [6,-9,5],
                     [9,9,9]])

kidx, sim = nearest_neighbor(v,candidates,2)
kidx, sim

(array([0, 2]),
 [np.float64(0.8320502943378437),
  np.float64(0.11470786693528087),
  np.float64(0.9486832980505138),
  np.float64(0.6527299120066193),
  np.float64(0.8164965809277259)])

### Probando la traducción y calculando el accuracy del modelo

* Calculando el accuracy como $$\text{accuracy}=\frac{\#(\text{correct predictions})}{\#(\text{total predictions})}$$

In [54]:
def test_vocabulary(X,Y,R):
    pred = np.dot(X, R)
    num_correct = 0

    for i in range(len(pred)):
        pred_idx, sim = nearest_neighbor(pred[i],Y)
        if pred_idx == i:
            num_correct +=1

    accuracy = num_correct/len(pred)
    return accuracy


In [56]:
y_hat = np.dot(X_train, R)
y_hat

array([[-0.02340129, -0.0032963 , -0.02684166, ...,  0.02744616,
         0.01715794, -0.00366401],
       [-0.01362437, -0.00129046, -0.05520523, ...,  0.01677211,
         0.0459886 ,  0.00717925],
       [-0.01245856,  0.0137361 , -0.01382703, ...,  0.02461993,
         0.05008422, -0.01362901],
       ...,
       [ 0.02177541, -0.05538198, -0.03846587, ..., -0.01560807,
         0.03598485, -0.02774162],
       [-0.01879526, -0.00433944,  0.03008367, ...,  0.06957914,
        -0.00162228, -0.03386664],
       [-0.0270601 , -0.01752797, -0.02028583, ...,  0.04419445,
         0.05666284,  0.01831575]], shape=(4932, 300))

In [57]:
#Probando el modelo
en_fr_test

{'torpedo': 'torpilles',
 'giovanni': 'giovanni',
 'chat': 'chat',
 'catholics': 'catholiques',
 'herald': 'herald',
 'chuck': 'chuck',
 'pit': 'fosse',
 'supplied': 'fournie',
 'optional': 'facultatives',
 'garrison': 'garnison',
 'sprint': 'sprint',
 'exile': 'exilés',
 'surprised': 'étonnée',
 'achievements': 'réalisations',
 'biblical': 'bibliques',
 'rebels': 'rebelles',
 'denis': 'denis',
 'geographical': 'géographique',
 'sit': 'sit',
 'alpine': 'alpine',
 'bills': 'factures',
 'glacier': 'glaciers',
 'binding': 'reliure',
 'indicating': 'indiquant',
 'estonia': 'estonie',
 'eating': 'manger',
 'saving': 'économiser',
 'chi': 'chi',
 'developer': 'développeurs',
 'indie': 'indie',
 'difficulties': 'difficultés',
 'doctrine': 'doctrine',
 'worn': 'porté',
 'fork': 'fourches',
 'simpson': 'simpson',
 'maintaining': 'maintenir',
 'theological': 'théologique',
 'upcoming': 'prochaines',
 'temporarily': 'momentanément',
 'hotels': 'hôtels',
 'edmonton': 'edmonton',
 'developments': '

In [59]:
X_test, Y_test = get_matrices(en_fr_test, fr_embeddings_subset, en_embeddings_subset)

In [60]:
X_test

array([[ 0.06396484,  0.40234375,  0.02319336, ..., -0.20898438,
        -0.22363281, -0.00683594],
       [-0.00363159,  0.15527344, -0.25390625, ..., -0.25195312,
        -0.296875  , -0.21582031],
       [ 0.17480469,  0.04345703,  0.24121094, ..., -0.16015625,
         0.1015625 ,  0.10058594],
       ...,
       [-0.08984375, -0.04394531, -0.06787109, ...,  0.05932617,
         0.15039062,  0.08496094],
       [-0.171875  ,  0.09814453, -0.07275391, ..., -0.23828125,
        -0.03222656, -0.21191406],
       [ 0.03662109,  0.03393555, -0.05517578, ..., -0.13964844,
        -0.04785156,  0.1015625 ]], shape=(1438, 300), dtype=float32)

In [61]:
Y_test

array([[-0.0272983 ,  0.0232864 , -0.0844774 , ...,  0.0174428 ,
         0.0160508 ,  0.0779456 ],
       [-0.00176859,  0.00113726, -0.0841218 , ...,  0.0269044 ,
         0.0529719 ,  0.0883998 ],
       [ 0.0114187 , -0.0113669 , -0.0265945 , ...,  0.0484328 ,
         0.0140038 ,  0.0631717 ],
       ...,
       [ 0.0496216 ,  0.0324685 , -0.0189056 , ...,  0.0695983 ,
         0.0402269 , -0.0755109 ],
       [-0.0322527 ,  0.0141267 , -0.0416719 , ...,  0.00357539,
         0.0874283 , -0.0216417 ],
       [-0.0577526 ,  0.00185741, -0.0752259 , ..., -0.0343118 ,
         0.0140071 , -0.0172464 ]], shape=(1438, 300), dtype=float32)

In [62]:
R

array([[-1.08811231e-02, -1.15382179e-02,  1.06349761e-04, ...,
         3.62303333e-03, -3.66357487e-03,  2.07701817e-03],
       [ 1.83240298e-02, -1.21513217e-03,  3.83943935e-03, ...,
         2.29956061e-02, -2.47546847e-03,  1.12509892e-03],
       [ 4.86021680e-03, -5.86435761e-04,  1.49689374e-02, ...,
        -4.45355132e-03, -6.88428530e-03,  4.33603814e-03],
       ...,
       [-2.08275863e-03,  5.39970785e-03, -1.84737780e-02, ...,
        -1.06562765e-02,  6.04376001e-03,  1.81758920e-03],
       [ 4.19663560e-03, -7.23999999e-03, -9.53285322e-04, ...,
        -7.28007045e-03, -1.24945350e-03, -4.23268660e-05],
       [ 6.90404170e-03, -1.33655245e-02, -4.34918391e-03, ...,
        -1.31628248e-02,  9.36894653e-03, -3.94386594e-03]],
      shape=(300, 300))

In [63]:
#Calculando el accuracy
acc = test_vocabulary(X_test,Y_test,R)

In [64]:
acc

0.5591098748261474

In [67]:
#Traducir laa palabra en frances en función a una palbra en inglés
def predict_fr_word(en_word, en_embedding, R):
    # 1.- Encontrar el vector en el espacio de embeddings en francés
    # 2.- Encontrar el vecino más cercano a ese vector
    # 3.- combetir el vector embedding a su palabra correspondiente (diccionario de embeddings?)
    # 4.- fr_word----> corresponde a la paalabra en francés
    
    return fr_word

In [66]:
 # predict_fr_word(['the', 'oil'], en_embedding_subset, R)
 # Frances = ['', '']